In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# --- 1. DATA PREPARATION ---
print("Loading data for the Master Robustness Showdown...")
df = pd.read_csv('wesad_final_normalized_features.csv')

ecg_cols = [c for c in df.columns if 'HRV' in c or 'ECG_Rate' in c]
gsr_cols = [c for c in df.columns if 'SCR' in c or 'EDA' in c]
fusion_cols = ecg_cols + gsr_cols

df['label_idx'] = df['label'].map({1: 0, 2: 1})
df = df[df['label_idx'].notnull()] 

subjects = df['subject_id'].unique()
noise_levels = [0.0, 0.5, 1.0, 1.5, 2.0]
master_results = []

print("Running simultaneous evaluation of Early and Late Fusion across noise sweeps...")

# --- 2. OUTER LOOP: ESCALATING NOISE LEVELS ---
for noise in noise_levels:
    actual_labels = []
    early_preds = []
    late_preds = []
    
    # --- 3. INNER LOOP: LOSO VALIDATION ---
    for test_sub in tqdm(subjects, desc=f"Noise Level {noise} std"):
        train_df = df[df['subject_id'] != test_sub]
        # Copy to ensure we don't permanently corrupt the test data
        test_df = df[df['subject_id'] == test_sub].copy() 
        
        y_train = train_df['label_idx']
        y_test = test_df['label_idx']
        
        # A. Train Early Fusion Model (All Features)
        model_early = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
        model_early.fit(train_df[fusion_cols], y_train)
        
        # B. Train Late Fusion Experts (Isolated Modalities)
        model_ecg = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42, eval_metric='logloss')
        model_gsr = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42, eval_metric='logloss')
        model_ecg.fit(train_df[ecg_cols], y_train)
        model_gsr.fit(train_df[gsr_cols], y_train)
        
        # --- 4. HARDWARE FAILURE SIMULATION ---
        if noise > 0:
            for col in ecg_cols:
                random_noise = np.random.normal(0, noise, test_df[col].shape)
                test_df[col] = test_df[col] + random_noise
                
        # --- 5. PREDICTIONS UNDER NOISE ---
        # Early Fusion evaluates the combined (but partially noisy) feature space
        pred_early = model_early.predict(test_df[fusion_cols])
        
        # Late Fusion evaluates separately and averages
        prob_ecg_noisy = model_ecg.predict_proba(test_df[ecg_cols])[:, 1]
        prob_gsr_clean = model_gsr.predict_proba(test_df[gsr_cols])[:, 1]
        prob_late = (prob_ecg_noisy + prob_gsr_clean) / 2.0
        pred_late = (prob_late >= 0.5).astype(int)
        
        actual_labels.extend(y_test.tolist())
        early_preds.extend(pred_early.tolist())
        late_preds.extend(pred_late.tolist())
        
    # Calculate global F1 for both architectures at this noise level
    f1_early = f1_score(actual_labels, early_preds)
    f1_late = f1_score(actual_labels, late_preds)
    
    # Store for the plot
    master_results.append({'Noise_Level': noise, 'Architecture': 'Early Fusion', 'F1_Score': f1_early})
    master_results.append({'Noise_Level': noise, 'Architecture': 'Late Fusion', 'F1_Score': f1_late})

# --- 6. RESULTS AND PLOTTING ---
results_df = pd.DataFrame(master_results)

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Plot both lines with distinct colors and markers
sns.lineplot(data=results_df, x='Noise_Level', y='F1_Score', hue='Architecture', 
             style='Architecture', markers=['o', 's'], dashes=False, 
             palette=['#1f77b4', '#9467bd'], linewidth=2.5, markersize=9)

plt.title('Architectural Robustness: Feature vs. Decision-Level Fusion Degradation', fontsize=14, pad=15)
plt.xlabel('Simulated Sensor Failure (ECG Gaussian Noise std)', fontsize=12)
plt.ylabel('Overall F1-Score', fontsize=12)
plt.ylim(0.75, 0.95)
plt.xticks(noise_levels)
plt.legend(title='Fusion Strategy', fontsize=11, title_fontsize=12)
plt.tight_layout()

plt.savefig('thesis_master_robustness_curve.png', dpi=300)
plt.show()

print("\nMaster architectural comparison complete. Plot saved as 'thesis_master_robustness_curve.png'.")